In [ ]:
# import the required programs - you only need to do this once upon loading the notebook

!pip install cantera
import cantera as ct
import numpy as np
gas = ct.Solution("/content/FFCM2_AME436.yaml") # choose thermodynamic data base

In [ ]:
# LH2 - LOX, SSME (RS-25 engine); T_chamber = 3569.5 K, P_star = 114.8 atm, P_exit = 0.17945 atm
h_LH2 = -1.89; MW_LH2 = 0.002; rho_LH2 = 71; DeltaP_LH2 = 7020   # units kcal/mole, kg/mole, kg/m^3, lbf/in^2
h_LO2 = -3.08; MW_LO2 = 0.032; rho_LO2 = 1140; DeltaP_LO2 = 8080  # units kcal/mole, kg/mole, kg/m^3, lbf/in^2
YH2 = 1/7; YO2 = 6/7
h_propellant = (YO2 * h_LO2/MW_LO2 + YH2 * h_LH2/MW_LH2) * 4184
eta_pump = 0.5
Pump_work = (1/eta_pump) * (YO2 * DeltaP_LO2/rho_LO2 + YH2 * DeltaP_LH2/rho_LH2) * 6895  # 6895 N/m^2 = 1 lbf/in^2
h_propellant = h_propellant - Pump_work
P_chamber = 203.7 * 101325                            # chamber pressure = 2997 psi
A_star = (np.pi/4)* 0.260**2                          # throat diameter = 0.260 m
AeOverAstar = 77.5                                    # area ratio Ae/A*
P_amb = 101325                                        # ambient pressure

In [ ]:
# compute chamber temperature

T_chamber = 3569.5                                      # combustion temperature guess - keep adjusting until h_error is small
mixture = f"H2:1 O2:6"                                  # atoms of reactants; use masses with gas.TPY or moles with gas.TPX
gas.TPY = T_chamber, P_chamber, mixture                 # react mixture at specified temperature and pressure
gas.equilibrate("TP")
# h_propellant = gas.h                                  # use this line ONLY for non-reacting propellants, otherwise comment out
h_chamber = gas.h; s_chamber = gas.s                    # calculate enthalpy and entropy
h_error = (h_propellant - h_chamber)/(0.5 * (h_propellant + h_chamber))
print(f"Fractional enthalpy error = {h_error:.6g}")     # adust T_chamber until h_error <<<< 1

Fractional enthalpy error = -4.08503e-05


In [ ]:
# expand from chamber condition (M << 1) to throat; this is needed to determine mass flow

s_star = s_chamber                                      # isentropic expansion from chamber to throat
P_star = 114.8 * 101325                                 # guess for chamber pressure; adjust until M_star = 1
gas.SP = s_star, P_star                                 # expand isentropically to target throat condition
gas.equilibrate("SP")                                   # this is a rocket - no frozen flow here!
h_star = gas.h                                          # get entalpy, sound speed, and density
c_star = gas.sound_speed
rho_star = gas.density
u_star = np.sqrt(2*(h_chamber - h_star))                # calculate velocity
mdot_star = rho_star * u_star * A_star                  # calculate mass flow - must be same at exit!
M_star = u_star/c_star                                  # calculate Mach number
print(f"Throat Mach number = {M_star:.6g}")             # is M = 1? if not, adjust P_star

Throat Mach number = 0.999642


In [ ]:
# expand from throat to exit for a given Ae/A*

s_exit = s_star                                         # isentropic expansion from throat to exit
A_exit = A_star * AeOverAstar                           # exit area
P_exit = 0.17945 * 101325                               # guess for exit pressure; adjust until mdot_exit = mdot_star
gas.SP = s_exit, P_exit                                 # expand isentropically to target throat condition
gas.equilibrate("SP")                                   # this is a rocket - no frozen flow here!
h_exit = gas.h                                          # get entalpy, sound speed, and density
c_exit = gas.sound_speed
rho_exit = gas.density
u_exit = np.sqrt(2*(h_chamber - h_exit))
mdot_exit = rho_exit * A_exit * u_exit                  # calculate exit mass flow - must be same as throat!
mdot_error = (mdot_exit - mdot_star)/(0.5 * (mdot_exit + mdot_star))
print(f"mdot_error = {mdot_error:.8g}")                 # adjust P_exit until mdot_error is <<<< 1
T_exit = gas.T                                          # temperature and Mach number at exit not needed for calculations
M_exit = u_exit/c_exit                                  # ... but nice to know anyway

mdot_error = 9.5002893e-06


In [ ]:
# Calculate and print results

Thrust = mdot_exit * u_exit + (P_exit-P_amb) * A_exit   # calculate Thrust
Isp = Thrust/(mdot_exit * 9.806)                        # ... and Isp
print(f"Exit Mach number = {M_exit:.4g} \tExit T = {T_exit:.4g} K \tThrust = {Thrust:.4g} N \tIsp = {Isp:.4g} seconds \tmdot = {mdot_exit:.4g} kg/s")

# Print mole fractions in exhaust
species_names = gas.species_names                       # get all species names and their mole fractions
mole_fractions = gas.X
# Combine names & mole fractions (name, value) then sort descending order of mole fraction
species_data = sorted(zip(species_names, mole_fractions), key=lambda x: x[1], reverse=True)
print()
print(f"{'Species':<15} | {'Mole Fraction':<15}")
print("-" * 35)
for name, fraction in species_data[:10]:                 # print only 10 most abundant species
    if fraction > 1e-9:                                  # ... and only print species with significant concentrations
        print(f"{name:<15} | {fraction:.6e}")

Exit Mach number = 4.691 	Exit T = 1170 K 	Thrust = 1.73e+06 N 	Isp = 371.9 seconds 	mdot = 474.2 kg/s

Species         | Mole Fraction  
-----------------------------------
H2O             | 7.560472e-01
H2              | 2.439527e-01
H               | 1.287751e-07
OH              | 3.331188e-09
